In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import operator

In [2]:
load_dotenv()

True

In [9]:
model = ChatGroq(model="openai/gpt-oss-120b")

In [19]:
class EveluationSchema(BaseModel):

    feedback: str = Field(description="detailed Feedback for the essay")
    score: float = Field(description="Score out of 10", ge=0, le=10)

In [20]:
structured_model = model.with_structured_output(EveluationSchema)

In [21]:
essay=""""
India in the Age of AI
As the world enters a transformative era defined by artificial intelligence (AI), India stands at a critical juncture — one where it can either emerge as a global leader in AI innovation or risk falling behind in the technology race. The age of AI brings with it immense promise as well as unprecedented challenges, and how India navigates this landscape will shape its socio-economic and geopolitical future.

India's strengths in the AI domain are rooted in its vast pool of skilled engineers, a thriving IT industry, and a growing startup ecosystem. With over 5 million STEM graduates annually and a burgeoning base of AI researchers, India possesses the intellectual capital required to build cutting-edge AI systems. Institutions like IITs, IIITs, and IISc have begun fostering AI research, while private players such as TCS, Infosys, and Wipro are integrating AI into their global services. In 2020, the government launched the National AI Strategy (AI for All) with a focus on inclusive growth, aiming to leverage AI in healthcare, agriculture, education, and smart mobility.

One of the most promising applications of AI in India lies in agriculture, where predictive analytics can guide farmers on optimal sowing times, weather forecasts, and pest control. In healthcare, AI-powered diagnostics can help address India’s doctor-patient ratio crisis, particularly in rural areas. Educational platforms are increasingly using AI to personalize learning paths, while smart governance tools are helping improve public service delivery and fraud detection.

However, the path to AI-led growth is riddled with challenges. Chief among them is the digital divide. While metropolitan cities may embrace AI-driven solutions, rural India continues to struggle with basic internet access and digital literacy. The risk of job displacement due to automation also looms large, especially for low-skilled workers. Without effective skilling and re-skilling programs, AI could exacerbate existing socio-economic inequalities.

Another pressing concern is data privacy and ethics. As AI systems rely heavily on vast datasets, ensuring that personal data is used transparently and responsibly becomes vital. India is still shaping its data protection laws, and in the absence of a strong regulatory framework, AI systems may risk misuse or bias.

To harness AI responsibly, India must adopt a multi-stakeholder approach involving the government, academia, industry, and civil society. Policies should promote open datasets, encourage responsible innovation, and ensure ethical AI practices. There is also a need for international collaboration, particularly with countries leading in AI research, to gain strategic advantage and ensure interoperability in global systems.

India’s demographic dividend, when paired with responsible AI adoption, can unlock massive economic growth, improve governance, and uplift marginalized communities. But this vision will only materialize if AI is seen not merely as a tool for automation, but as an enabler of human-centered development.

"""

In [26]:
prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {essay}'
structured_model.invoke(prompt)

EveluationSchema(feedback='The essay is clear, well‑structured and demonstrates a strong command of English. It opens with a compelling hook, follows a logical progression of ideas, and uses appropriate terminology for the subject. Paragraphs are cohesive, transitions smooth, and the argument is supported with concrete examples (e.g., AI in agriculture, healthcare, education). Grammar and punctuation are largely correct, and the vocabulary is varied without being overly complex. Minor issues include a few instances of repetition (e.g., “AI‑led growth” and “AI‑driven solutions” close together) and occasional wordiness that could be tightened for sharper impact. Overall, the piece reads professionally and would be suitable for a policy brief or editorial.\n\nScore: 8.5/10', score=8.5)

In [27]:
# Lets define our workflow
# Create a state
class UPSEState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_score: Annotated[list[float], operator.add]  # Suppose we get 3 individual score form above 3 nodes and store it like [8,7,6], problem is that all values are generate at a time parallally, and  all values store in this variable so it can be over right (possibility). so reduce this problem we use reducer function , and reducer function is add.
    avg_score: float

In [28]:
# evaluate_language
def evaluate_language(state: UPSEState):

    prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'

    output = structured_model.invoke(prompt)

    return {"language_feedback": output.feedback,"individual_score": [output.score]}


In [29]:
# evaluate_analysis
def evaluate_analysis(state: UPSEState):

    prompt = f'Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'

    output = structured_model.invoke(prompt)

    return {"analysis_feedback": output.feedback,"individual_score": [output.score]}


In [30]:
# evaluate_thought
def evaluate_thought(state: UPSEState):

    prompt = f'Evaluate the clarity of thought in the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'

    output = structured_model.invoke(prompt)

    return {"clarity_feedback": output.feedback,"individual_score": [output.score]}


In [31]:
def final_evaluation(state: UPSEState):

    # Summary feedback
    prompt = f'Based on the following feedbacks create a summarized feedback \n language feedback - {state["language_feedback"]} \n depth of analysis feedback - {state["analysis_feedback"]} \n clarity of thought feedback - {state["clarity_feedback"]}'
    overall_feedback = model.invoke(prompt).content

    # Calculate average score
    average_score = sum(state["individual_score"])/len(state["individual_score"])

    return {"overall_feedback": overall_feedback, "avg_score": average_score}

In [34]:
# Build a grapg

graph = StateGraph(UPSEState)

# Nodes
graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)

# Edges
graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')

graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')

graph.add_edge( 'final_evaluation', END)

workflow = graph.compile()
# workflow


In [38]:
essay2 = """India and AI Time

Now world change very fast because new tech call Artificial Intel… something (AI). India also want become big in this AI thing. If work hard, India can go top. But if no careful, India go back.

India have many good. We have smart student, many engine-ear, and good IT peoples. Big company like TCS, Infosys, Wipro already use AI. Government also do program “AI for All”. It want AI in farm, doctor place, school and transport.

In farm, AI help farmer know when to put seed, when rain come, how stop bug. In health, AI help doctor see sick early. In school, AI help student learn good. Government office use AI to find bad people and work fast.

But problem come also. First is many villager no have phone or internet. So AI not help them. Second, many people lose job because AI and machine do work. Poor people get more bad.

One more big problem is privacy. AI need big big data. Who take care? India still make data rule. If no strong rule, AI do bad.

India must all people together – govern, school, company and normal people. We teach AI and make sure AI not bad. Also talk to other country and learn from them.

If India use AI good way, we become strong, help poor and make better life. But if only rich use AI, and poor no get, then big bad thing happen.

So, in short, AI time in India have many hope and many danger. We must go right road. AI must help all people, not only some. Then India grow big and world say "good job India"."""

In [40]:
intial_state = {
    "essay": essay
}

workflow.invoke(intial_state)

{'essay': '"\nIndia in the Age of AI\nAs the world enters a transformative era defined by artificial intelligence (AI), India stands at a critical juncture — one where it can either emerge as a global leader in AI innovation or risk falling behind in the technology race. The age of AI brings with it immense promise as well as unprecedented challenges, and how India navigates this landscape will shape its socio-economic and geopolitical future.\n\nIndia\'s strengths in the AI domain are rooted in its vast pool of skilled engineers, a thriving IT industry, and a growing startup ecosystem. With over 5 million STEM graduates annually and a burgeoning base of AI researchers, India possesses the intellectual capital required to build cutting-edge AI systems. Institutions like IITs, IIITs, and IISc have begun fostering AI research, while private players such as TCS, Infosys, and Wipro are integrating AI into their global services. In 2020, the government launched the National AI Strategy (AI 